# 22. Frozen screen of transformer backbones

Which transformer should be fine-tuned as a new ensemble member? Before spending GPU hours on five folds,
we repeat the section-3 baseline protocol with each candidate as a **frozen** feature extractor:
pooled features → standardised logistic regression per label → 5-fold out-of-fold (`folds.csv`).

The photos are 450×375. Patch-14 ViTs get **448×378** (the whole photo at about native scale, ~5 px padding),
SigLIP (patch 16) gets 448×368, and Swin needs its native 384×384 square. ConvNeXt-base at the current 384×384
letterbox is the CNN reference under the same protocol. Features were extracted with fp16 autocast on MPS
(cosine similarity to fp32 features ≥ 0.99999 on a 64-image check).

Extra column, **kNN AP**: for each named pet, the cosine similarity to the nearest photo of that pet in the
other folds, with no classifier at all. It measures how well the raw features recognise *one individual animal*.

Code: [stage2/vit_screen.py](../stage2/vit_screen.py). Embeddings are cached in `artifacts/vit_screen/` (ignored);
OOF probabilities and summaries are in [experiments/vit_screen/](../experiments/vit_screen/).
Terminal run: `python scripts/run_vit_screen.py`.

## Selection rule (fixed before the results)

1. **Primary: mean AP of the three named pets** (threshold-free; Cat/Dog are ≈1.0 for every model and cannot separate them).
2. **Secondary: fair macro F1** with per-label thresholds chosen on the other folds.
3. Differences below **0.005** in both are treated as a tie (≈ 2 named-pet photos). A tie goes to the cheaper model.
4. Only models that fit five-fold fine-tuning on a T4 can be the fine-tuning pick: **DINOv2-L is screened but not fine-tuned**.
   If it wins clearly, its frozen LogReg is a candidate ensemble member on its own.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RUN_SCREEN = os.environ.get("STAGE2_RUN_SCREEN", "0") == "1"  # embedding takes ~2.5 h on an M-series GPU

In [ ]:
from stage2.vit_screen import NAMED_PETS, SCREEN_ORDER, format_table, run_screen, summary_row

if RUN_SCREEN:
    run_screen(ROOT, device="mps", half=True)

In [ ]:
FINE_TUNABLE = {"dinov2_s", "dinov2_b", "eva02_b", "siglip_b", "swin_b"}
GMACS = {"convnext_b_ref": 45.1, "dinov2_s": 18.6, "dinov2_b": 74.2, "dinov2_l": 262.9, "eva02_b": 73.9, "siglip_b": 55.2, "swin_b": 47.1}

rows = []
for key in SCREEN_ORDER:
    path = ROOT / "experiments" / "vit_screen" / key / "summary.json"
    if path.exists():
        row = summary_row(key, json.loads(path.read_text()))
        rows.append({"model": key, "GMACs": GMACS[key],
                     "pet AP": sum(row[f"AP {pet}"] for pet in NAMED_PETS) / 3,
                     **{k: v for k, v in row.items() if k != "model"}})
rows.sort(key=lambda row: -row["pet AP"])
print(format_table(rows))

In [ ]:
TIE = 0.005
pool = [row for row in rows if row["model"] in FINE_TUNABLE]
best_ap = max(row["pet AP"] for row in pool)
best_f1 = max(row["F1 fair"] for row in pool)
tied = [row for row in pool if best_ap - row["pet AP"] < TIE and best_f1 - row["F1 fair"] < TIE]
pick = min(tied, key=lambda row: row["GMACs"])["model"] if tied else pool[0]["model"]
print("Within the tie margin on both metrics:", [row["model"] for row in tied])
print("Fine-tuning pick:", pick)